In [ ]:
import os
from pathlib import Path
import networkx as nx
import pandas as pd
from pydantic import BaseModel
from openai import OpenAI

PROJECT_DIR = Path(os.path.abspath("../.."))

MODEL = "google/gemma-4-31b-it"
client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key="YOUR_API_KEY_HERE")

OUT = PROJECT_DIR / "kcs_relationships.csv"
df = pd.read_csv(PROJECT_DIR / "kcs.csv")
df['level_n'] = df['level'].str[1:].astype(int)
df['key'] = df['id'] + "__" + df['level']
targets = df.to_dict('records')
cand_map = {t['key']: [c for c in targets if c['level_n'] <= t['level_n'] and c['id'] != t['id']] for t in targets}

In [ ]:
PROMPT_TEMPLATE = """Você é um especialista em educação e aprendizagem, atuando no contexto da Olimpíada Brasileira de Robótica (OBR).

Tarefa: Identificar relações de pré-requisito entre componentes de conhecimento (KCs), relevantes exclusivamente para o escopo da OBR.

KC-alvo:
{target}

KCs candidatos:
{cands}

Pergunta:
O domínio do KC candidato é necessário ou relevante para o KC-alvo, considerando o contexto da OBR?

Critérios:
- Valores próximos de 1 indicam que o domínio do KC candidato é necessário ou relevante para o KC-alvo.
- Valores próximos de 0 indicam que o KC candidato não é necessário ou relevante para o KC-alvo.

Responda APENAS em JSON: {{"target": "{tid}", "prerequisites": [{{"explanation": "...", "id": "...", "weight": 0.0-1.0}}]}}"""

In [3]:
LEVELS = {
    "N0": "1º ano (Ensino Fundamental)",
    "N1": "2º e 3º ano (Ensino Fundamental)",
    "N2": "4º e 5º ano (Ensino Fundamental)",
    "N3": "6º e 7º ano (Ensino Fundamental)",
    "N4": "8º e 9º ano (Ensino Fundamental)",
    "N5": "Ensino Médio e Técnico",
    "N5-F1": "Ensino Médio e Técnico",
    "N5-F2": "Ensino Médio e Técnico",
}

def format_kc(k):
    return f"- ID: {k['id']}\n- Nível: {LEVELS.get(k['level'], '')}\n- Assunto: {k['subject']}\n- Descrição: {k['description']}"

def build_prompt(t):
    return PROMPT_TEMPLATE.format(tid=t['id'], target=format_kc(t), cands="\n\n".join(format_kc(c) for c in cand_map[t['key']]))

In [ ]:
class Prerequisite(BaseModel):
    id: str
    weight: float
    explanation: str

class PrereqResponse(BaseModel):
    target: str
    prerequisites: list[Prerequisite]

def call_llm(prompt):
    r = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=PrereqResponse,
    )
    return r.choices[0].message.parsed

def done_keys():
    return set(pd.read_csv(OUT)['c2']) if os.path.exists(OUT) else set()

def append_rows(rows):
    if rows:
        pd.DataFrame(rows).to_csv(OUT, mode='a', header=not os.path.exists(OUT), index=False)

G = nx.DiGraph()
if os.path.exists(OUT):
    G.add_edges_from(pd.read_csv(OUT).dropna(subset=['c1','c2'])[['c1','c2']].values)

In [ ]:
processed = done_keys()
for t in targets:
    if t['id'] in processed: continue
    prompt = build_prompt(t)
    print(prompt)
    try: prereqs = call_llm(prompt).prerequisites
    except Exception as e: print(t['key'], e); continue
    rows = []
    for p in prereqs:
        if p.id == t['id']: continue
        if G.has_node(t['id']) and G.has_node(p.id) and nx.has_path(G, t['id'], p.id): continue
        G.add_edge(p.id, t['id'], weight=p.weight)
        rows.append({"c1": p.id, "c2": t['id'], "weight": p.weight, "explanation": p.explanation})
    append_rows(rows)

Você é um especialista em educação e aprendizagem, atuando no contexto da Olimpíada Brasileira de Robótica (OBR).

Tarefa: Identificar relações de pré-requisito entre componentes de conhecimento (KCs), relevantes exclusivamente para o escopo da OBR.

KC-alvo:
- ID: KC001
- Nível: 1º ano (Ensino Fundamental)
- Assunto: Língua Portuguesa
- Descrição: Leitura e compreensão de listas, instruções, agendas, calendários, avisos, convites, história em quadrinhos e tirinhas, parlendas, trava-línguas, receitas, relatos, entrevistas, anúncios e notícias.

KCs candidatos:
- ID: KC002
- Nível: 1º ano (Ensino Fundamental)
- Assunto: Língua Portuguesa
- Descrição: Conhecer, diferenciar e relacionar letras em formato imprensa e cursiva, maiúsculas e minúsculas.

- ID: KC003
- Nível: 1º ano (Ensino Fundamental)
- Assunto: Língua Portuguesa
- Descrição: Segmentar oralmente palavras em sílabas.

- ID: KC004
- Nível: 1º ano (Ensino Fundamental)
- Assunto: Língua Portuguesa
- Descrição: Comparar palavras, 

In [6]:
result = pd.read_csv(OUT)
target_key = "KC001"
desc = df.set_index('id')['description'].to_dict()
prereqs = result[(result['c2'] == target_key) & (result['weight'] > 0)]
print(f"- id: {target_key}\n- description: {desc[target_key]}")
for _, row in prereqs.iterrows():
    print(f"\t* id: {row['c1']}\n\t* weight: {row['weight']:.2f}\n\t* description: {desc[row['c1']]}\n")

- id: KC001
- description: Leitura e compreensão de listas, instruções, agendas, calendários, avisos, convites, história em quadrinhos e tirinhas, parlendas, trava-línguas, receitas, relatos, entrevistas, anúncios e notícias.
	* id: KC002
	* weight: 0.90
	* description: Conhecer, diferenciar e relacionar letras em formato imprensa e cursiva, maiúsculas e minúsculas.

	* id: KC005
	* weight: 0.70
	* description: Identificar outros sinais no texto além das letras, como pontos finais, de interrogação e exclamação e seus efeitos na entonação.

	* id: KC006
	* weight: 0.50
	* description: Identificar elementos de uma narrativa lida ou escutada, incluindo personagens, enredo, tempo e espaço.

